In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
import os

llm = ChatOpenAI(
    model="deepseek-chat",
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = "https://api.deepseek.com",
    temperature = 0,
)

In [16]:
from langchain_core.tools import tool

def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

def web_search(query: str) -> str:
    """Search the web for information."""
    return (
        "Here are the headcounts for each of the FAANG companies in 2024:\n"
        "1. **Facebook (Meta)**: 67,317 employees.\n"
        "2. **Apple**: 164,000 employees.\n"
        "3. **Amazon**: 1,551,000 employees.\n"
        "4. **Netflix**: 14,000 employees.\n"
        "5. **Google (Alphabet)**: 181,269 employees."
    )

math_agent = create_react_agent(
    model=llm,
    tools=[add, multiply],
    name="math_expert",
    prompt="You are a math expert. Always use one tool at a time."
)

research_agent = create_react_agent(
    model=llm,
    tools=[web_search],
    name="research_expert",
    prompt="You are a world class researcher with access to web search. Do not do any math."
)

C:\Users\49172\AppData\Local\Temp\ipykernel_37756\422722641.py:22: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  math_agent = create_react_agent(
C:\Users\49172\AppData\Local\Temp\ipykernel_37756\422722641.py:29: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(


In [17]:
from langgraph_supervisor import create_supervisor

workflow = create_supervisor(
    [research_agent, math_agent],
    model=llm,
    prompt=("You are a team supervisor managing a research expert and a math expert. "
    "For current events, use research_agent. "
    "FOr math problems, use math_agent.")
)

In [21]:
app = workflow.compile()
config = {
    "configurable": {
        "thread_id": "301"
    }
}
result = app.invoke({
    "messages":[
        {
            "role": "user",
            "content": "What's the combined headcount of the FANNG companies in 2024?"
        }
    ]
}, config=config)

In [19]:
result

{'messages': [HumanMessage(content="What's the combined headcount of the FANNG companies in 2024?", additional_kwargs={}, response_metadata={}, id='5f486c4f-c70d-42b8-bc23-a41773c4ff37'),
  AIMessage(content="I'll help you find the combined headcount of the FANNG companies in 2024. This requires research on current employment data for these major tech companies. Let me transfer you to our research expert who can gather the most up-to-date information.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 378, 'total_tokens': 459, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 378}, 'model_provider': 'openai', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': '370d3b35-5cab-40fb-a5b9-46528b89749d', 'finish_reason': 'tool_calls', 'logprobs': None}, name='supervisor', 